# 🔬 Python Code Bug Detection System — Research Quality v2

**Improvements over v1:**
- ✅ Real data: CodeSearchNet (10,000+ real GitHub Python functions)
- ✅ 8-class bug type classification (not just clean vs buggy)
- ✅ Attention-based suspicious line highlighting
- ✅ Fix suggestions per bug type
- ✅ Two CodeBERT models: binary detector + type classifier
- ✅ Research-quality visualisation dashboard

## Before you start
**Enable GPU:** Runtime → Change runtime type → **T4 GPU** → Save

Then run cells **top to bottom**.

## Cell 1 — Install packages

In [1]:
!pip install transformers datasets gradio torch scikit-learn matplotlib seaborn --quiet
print('✅ All packages installed!')

✅ All packages installed!


## Cell 2 — Imports and GPU check

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import numpy as np
import random
import tokenize
import io
import re
from collections import Counter, defaultdict

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU found. Go to Runtime > Change runtime type > T4 GPU')

Using device: cpu
⚠️  No GPU found. Go to Runtime > Change runtime type > T4 GPU


## Cell 3 — Load real GitHub functions from CodeSearchNet

This streams **real Python functions** from GitHub. No manual examples.
First run takes ~2-3 minutes. Increase `num_samples` for a larger dataset.

In [3]:
def load_codesearchnet(num_samples=6000):   # ✅ reduced from 10000 → 6000
    print(f'Streaming {num_samples} real Python functions from CodeSearchNet...')
    ds = load_dataset(
        'code-search-net/code_search_net',
        'python',
        split='train',
        streaming=True
    )
    clean_codes = []
    for sample in ds:
        if len(clean_codes) >= num_samples:
            break
        code = sample['whole_func_string']
        if len(code) < 50 or len(code) > 2000 or 'def ' not in code:
            continue
        clean_codes.append(code)
        if len(clean_codes) % 1000 == 0:
            print(f'  Loaded {len(clean_codes)} valid functions...')

    print(f'Total real functions loaded: {len(clean_codes)}')
    return clean_codes

REAL_CLEAN_CODES = load_codesearchnet(num_samples=6000)
print('\n--- Example real GitHub function (first 400 chars) ---')
print(REAL_CLEAN_CODES[0][:400])

Streaming 6000 real Python functions from CodeSearchNet...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  Loaded 1000 valid functions...
  Loaded 2000 valid functions...
  Loaded 3000 valid functions...
  Loaded 4000 valid functions...
  Loaded 5000 valid functions...
  Loaded 6000 valid functions...
Total real functions loaded: 6000

--- Example real GitHub function (first 400 chars) ---
def __multiscale_gc_hi2lo_run(self):  # , pyed):
        """
        Run Graph-Cut segmentation with simplifiyng of high resolution multiscale graph.
        In first step is performed normal GC on low resolution data
        Second step construct finer grid on edges of segmentation from first
        step.
        There is no option for use without `use_boundary_penalties`
        """
        # f


## Cell 4 — Bug taxonomy (8 types) and injection functions

Based on real bug categories found in BugsInPy analysis papers.

| ID | Type | Description |
|----|------|-------------|
| 0 | clean | No bug |
| 1 | off_by_one | Loop/index error |
| 2 | wrong_operator | Wrong arithmetic or comparison |
| 3 | missing_return | Return value missing |
| 4 | wrong_variable | Wrong variable name used |
| 5 | type_error | Type mismatch |
| 6 | logic_inversion | Condition flipped |
| 7 | missing_init | Variable used before assignment |

In [4]:
BUG_TYPES = {
    0: 'clean', 1: 'off_by_one', 2: 'wrong_operator',
    3: 'missing_return', 4: 'wrong_variable', 5: 'type_error',
    6: 'logic_inversion', 7: 'missing_init'
}

BUG_DESCRIPTIONS = {
    1: 'Off-by-one error in loop or index',
    2: 'Wrong arithmetic or comparison operator',
    3: 'Missing or incorrect return statement',
    4: 'Wrong variable name used',
    5: 'Type mismatch or wrong conversion',
    6: 'Logic condition inverted',
    7: 'Variable used before proper initialization',
}

BUG_FIXES = {
    1: 'Check loop ranges and indices — range(n) not range(n+1), lst[0] not lst[1]',
    2: 'Review operator: likely == vs !=, > vs <, or + vs -',
    3: 'Add a return statement — function may fall through without returning the computed value',
    4: 'Check variable names — a variable from outer scope may have been used instead',
    5: 'Check types: int/str mismatch or wrong division type',
    6: 'Condition is inverted — try flipping the if/else or negating the condition',
    7: 'Initialize variable before the conditional block to ensure it always has a value',
}


def inject_off_by_one(code):
    lines = code.split('\n')
    for i, line in enumerate(lines):
        m = re.search(r'range\(([^)]+)\)', line)
        if m:
            lines[i] = line.replace(m.group(0), f'range({m.group(1).strip()}+1)', 1)
            return '\n'.join(lines), i
        if '[0]' in line:
            lines[i] = line.replace('[0]', '[1]', 1)
            return '\n'.join(lines), i
    return None, None


def inject_wrong_operator(code):
    swaps = [(' == ',' != '),(' != ',' == '),(' >= ',' > '),
             (' <= ',' < '),(' > ',' < '),(' + ',' - '),(' // ',' / ')]
    lines = code.split('\n')
    for i, line in enumerate(lines):
        if line.strip().startswith(('#','import','def ')): continue
        for old, new in swaps:
            if old in line:
                lines[i] = line.replace(old, new, 1)
                return '\n'.join(lines), i
    return None, None


def inject_missing_return(code):
    lines = code.split('\n')
    for i in range(len(lines)-1, -1, -1):
        s = lines[i].strip()
        if s.startswith('return ') and len(s) > 7:
            indent = len(lines[i]) - len(lines[i].lstrip())
            lines[i] = ' '*indent + '# BUG: ' + s
            return '\n'.join(lines), i
    return None, None


def inject_wrong_variable(code):
    skip = {'self','cls','None','True','False','i','j','k','n','x','y'}
    found = [v for v in re.findall(r'\b([a-z_][a-z_0-9]*)\s*=\s*(?!=)', code)
             if v not in skip and len(v) > 1]
    found = list(dict.fromkeys(found))
    if len(found) < 2:
        return None, None
    v1, v2 = found[0], found[1]
    lines = code.split('\n')
    for i, line in enumerate(lines):
        if v1 in line and f'{v1} =' not in line and f'{v1}=' not in line:
            lines[i] = re.sub(r'\b' + re.escape(v1) + r'\b', v2, line, count=1)
            return '\n'.join(lines), i
    return None, None


def inject_type_error(code):
    lines = code.split('\n')
    for i, line in enumerate(lines):
        if line.strip().startswith(('#','def ','return')): continue
        if ' // ' in line:
            lines[i] = line.replace(' // ', ' / ', 1)
            return '\n'.join(lines), i
        if '.append(' in line:
            lines[i] = line.replace('.append(', '.extend(', 1)
            return '\n'.join(lines), i
    return None, None


def inject_logic_inversion(code):
    lines = code.split('\n')
    for i, line in enumerate(lines):
        s = line.strip()
        if s.startswith('if ') or s.startswith('elif '):
            kw = 'if ' if s.startswith('if ') else 'elif '
            indent = len(line) - len(line.lstrip())
            if 'not ' not in s:
                cond = s[len(kw):s.rfind(':')]
                lines[i] = ' '*indent + kw + 'not (' + cond + '):'
                return '\n'.join(lines), i
    return None, None


def inject_missing_init(code):
    pat = re.compile(r'^(\s+)([a-z_]\w*)\s*=\s*(\[\]|\{\}|\(\)|0|""|None|False|True)\s*$')
    lines = code.split('\n')
    for i, line in enumerate(lines):
        m = pat.match(line)
        if m:
            var = m.group(2)
            if var in '\n'.join(lines[i+1:]):
                lines[i] = m.group(1) + f'# BUG: removed init of {var}'
                return '\n'.join(lines), i
    return None, None


INJECTORS = [
    (inject_off_by_one, 1), (inject_wrong_operator, 2),
    (inject_missing_return, 3), (inject_wrong_variable, 4),
    (inject_type_error, 5), (inject_logic_inversion, 6),
    (inject_missing_init, 7),
]

def create_bug(code):
    injectors = INJECTORS.copy()
    random.shuffle(injectors)
    for fn, bug_id in injectors:
        try:
            buggy, line_num = fn(code)
            if buggy and buggy != code:
                return buggy, bug_id, line_num
        except Exception:
            continue
    return None, None, None

# Quick test
b, t, l = create_bug(REAL_CLEAN_CODES[0])
if b:
    print(f'Test injection OK — type: {BUG_TYPES[t]}, line: {l}')
print('Bug injection system ready.')

Test injection OK — type: wrong_variable, line: 16
Bug injection system ready.


## Cell 5 — Build dataset from real functions

In [5]:
def build_dataset(clean_codes):
    codes, labels, bug_types, bug_lines = [], [], [], []
    skipped = 0
    for i, code in enumerate(clean_codes):
        codes.append(code); labels.append(0); bug_types.append(0); bug_lines.append(-1)
        buggy, btype, bline = create_bug(code)
        if buggy:
            codes.append(buggy); labels.append(1)
            bug_types.append(btype); bug_lines.append(bline if bline else -1)
        else:
            skipped += 1
        if (i+1) % 1000 == 0:
            print(f'  Processed {i+1}/{len(clean_codes)}...')
    print(f'\nDataset ready: {len(codes)} samples ({labels.count(0)} clean, {labels.count(1)} buggy)')
    print(f'Bug type distribution: {dict(Counter(bug_types))}')
    return codes, labels, bug_types, bug_lines

print('Building dataset...')
codes, labels, bug_types, bug_lines = build_dataset(REAL_CLEAN_CODES)

Building dataset...
  Processed 1000/6000...
  Processed 2000/6000...
  Processed 3000/6000...
  Processed 4000/6000...
  Processed 5000/6000...
  Processed 6000/6000...

Dataset ready: 11632 samples (6000 clean, 5632 buggy)
Bug type distribution: {0: 6000, 4: 1217, 3: 2196, 6: 996, 5: 158, 2: 647, 1: 230, 7: 188}


## Cell 6 — Train / Val / Test split (80 / 10 / 10)

In [6]:
X_tmp, X_test, y_tmp, y_test, bt_tmp, bt_test, bl_tmp, bl_test = train_test_split(
    codes, labels, bug_types, bug_lines,
    test_size=0.10, random_state=SEED, stratify=labels
)
X_train, X_val, y_train, y_val, bt_train, bt_val, bl_train, bl_val = train_test_split(
    X_tmp, y_tmp, bt_tmp, bl_tmp,
    test_size=0.111, random_state=SEED, stratify=y_tmp
)
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

Train: 9,306 | Val: 1,162 | Test: 1,164


## Cell 7 — CodeBERT tokenizer and Dataset classes

In [7]:
print('Loading CodeBERT tokenizer...')
tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')
print('Tokenizer loaded.')

class BugDataset(Dataset):
    """Binary: clean=0, buggy=1"""
    def __init__(self, codes, labels, tokenizer, max_len=128):  # ✅ 256 → 128
        self.codes = codes; self.labels = labels
        self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.codes[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

class BugTypeDataset(Dataset):
    """8-class: 0=clean, 1-7=bug types"""
    def __init__(self, codes, bug_types, tokenizer, max_len=128):  # ✅ 256 → 128
        self.codes = codes; self.bug_types = bug_types
        self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.bug_types)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.codes[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.bug_types[idx], dtype=torch.long)}

BATCH_SIZE = 8      # ✅ reduced from 16 → 8
MAX_LEN    = 128    # ✅ reduced from 256 → 128

train_ds = BugDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = BugDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = BugDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

Loading CodeBERT tokenizer...


Tokenizer loaded.
Train batches: 1164 | Val: 146 | Test: 146


## Cell 8 — Training helper functions

In [8]:
def train_one_epoch(model, loader, optimizer, scheduler, epoch, total):
    model.train()
    total_loss, correct, n = 0, 0, 0
    for step, batch in enumerate(loader):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=lbls)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
        correct += (out.logits.argmax(1) == lbls).sum().item()
        n += len(lbls)
        if (step+1) % 100 == 0:
            print(f'  Epoch {epoch}/{total} Step {step+1}/{len(loader)} | loss={out.loss.item():.4f} acc={correct/n:.4f}')
    return total_loss/len(loader), correct/n

def evaluate_model(model, loader):
    model.eval()
    total_loss, correct, n = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
            total_loss += out.loss.item()
            preds = out.logits.argmax(1)
            probs = F.softmax(out.logits, dim=1)[:, -1]
            correct += (preds == lbls).sum().item()
            n += len(lbls)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return total_loss/len(loader), correct/n, all_preds, all_labels, all_probs

print('Training functions ready.')

Training functions ready.


## Cell 9 — Train CodeBERT binary classifier (clean vs buggy)
Expected time: **~12–20 min on T4 GPU**

In [9]:
print('Loading CodeBERT for binary classification...')
binary_model = RobertaForSequenceClassification.from_pretrained(
    'microsoft/codebert-base', num_labels=2).to(device)
print(f'Parameters: {sum(p.numel() for p in binary_model.parameters()):,}')

BINARY_EPOCHS = 3
bin_opt = optim.AdamW(binary_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * BINARY_EPOCHS
bin_sched = get_linear_schedule_with_warmup(bin_opt, total_steps//10, total_steps)

bin_history = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
best_bin_acc = 0

for epoch in range(1, BINARY_EPOCHS+1):
    print(f'\n{"="*55}\nEpoch {epoch}/{BINARY_EPOCHS}\n{"="*55}')
    tr_loss, tr_acc = train_one_epoch(binary_model, train_loader, bin_opt, bin_sched, epoch, BINARY_EPOCHS)
    va_loss, va_acc, _, _, _ = evaluate_model(binary_model, val_loader)
    bin_history['train_loss'].append(tr_loss); bin_history['val_loss'].append(va_loss)
    bin_history['train_acc'].append(tr_acc);   bin_history['val_acc'].append(va_acc)
    print(f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Val Loss: {va_loss:.4f} Acc: {va_acc:.4f}')
    if va_acc > best_bin_acc:
        best_bin_acc = va_acc
        binary_model.save_pretrained('codebert_binary')
        tokenizer.save_pretrained('codebert_binary')
        print(f'  ✅ Saved best model (val acc={va_acc:.4f})')

print(f'\nBest binary val accuracy: {best_bin_acc:.4f}')

Loading CodeBERT for binary classification...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 124,647,170

Epoch 1/3


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

## Cell 10 — Evaluate binary model on test set

In [ ]:
# Reload with eager attention to match Cell 13 inference behavior
binary_model = RobertaForSequenceClassification.from_pretrained(
    'codebert_binary',
    attn_implementation="eager"   # ✅ must match Cell 13
).to(device)

_, _, test_preds, test_labels_out, test_probs = evaluate_model(binary_model, test_loader)

print('='*55)
print('BINARY BUG DETECTION — TEST SET RESULTS')
print('='*55)
print(f'Accuracy : {accuracy_score(test_labels_out, test_preds):.4f}')
print(f'F1 Score : {f1_score(test_labels_out, test_preds):.4f}')
print(f'Precision: {precision_score(test_labels_out, test_preds):.4f}')
print(f'Recall   : {recall_score(test_labels_out, test_preds):.4f}')
print()
print(classification_report(test_labels_out, test_preds, target_names=['Clean','Buggy']))

# ── Auto-compute optimal threshold from val set ────────────────────────────
_, _, val_preds_raw, val_labels_raw, val_probs_raw = evaluate_model(binary_model, val_loader)

best_thresh, best_f1 = 0.5, 0.0
for t in [i/100 for i in range(10, 70)]:           # sweep 0.10 → 0.69
    preds_t = [1 if p > t else 0 for p in val_probs_raw]
    f1 = f1_score(val_labels_raw, preds_t, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

OPTIMAL_THRESHOLD = round(best_thresh, 2)
print(f'\n✅ Auto-computed optimal threshold: {OPTIMAL_THRESHOLD} (val F1={best_f1:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cm = confusion_matrix(test_labels_out, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Clean','Buggy'], yticklabels=['Clean','Buggy'], ax=axes[0])
axes[0].set_title('Binary Model — Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

ep = range(1, BINARY_EPOCHS+1)
axes[1].plot(ep, bin_history['train_acc'], 'o-', label='Train')
axes[1].plot(ep, bin_history['val_acc'],   's-', label='Val')
axes[1].set_title('Binary Model — Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('binary_results.png', dpi=150); plt.show()

## Cell 11 — Train CodeBERT bug TYPE classifier (8 classes)
This is the research-quality upgrade — classifying *which* bug type is present.

Expected time: **~12–20 min on T4 GPU**

In [ ]:
# ✅ Free binary_model from GPU before loading type_model
del binary_model
torch.cuda.empty_cache()
import gc; gc.collect()
print("GPU cleared. Loading type model...")

type_train_ds = BugTypeDataset(X_train, bt_train, tokenizer, MAX_LEN)
type_val_ds   = BugTypeDataset(X_val,   bt_val,   tokenizer, MAX_LEN)
type_test_ds  = BugTypeDataset(X_test,  bt_test,  tokenizer, MAX_LEN)

type_train_loader = DataLoader(type_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=2, pin_memory=True)
type_val_loader   = DataLoader(type_val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=2, pin_memory=True)
type_test_loader  = DataLoader(type_test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=2, pin_memory=True)

print('Loading CodeBERT for 8-class type classification...')
type_model = RobertaForSequenceClassification.from_pretrained(
    'microsoft/codebert-base', num_labels=8).to(device)

TYPE_EPOCHS = 3
type_opt   = optim.AdamW(type_model.parameters(), lr=2e-5, weight_decay=0.01)
type_total = len(type_train_loader) * TYPE_EPOCHS
type_sched = get_linear_schedule_with_warmup(type_opt, type_total//10, type_total)

type_history  = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
best_type_acc = 0

for epoch in range(1, TYPE_EPOCHS+1):
    print(f'\n{"="*55}\nEpoch {epoch}/{TYPE_EPOCHS}\n{"="*55}')
    tr_loss, tr_acc = train_one_epoch(type_model, type_train_loader,
                                      type_opt, type_sched, epoch, TYPE_EPOCHS)
    va_loss, va_acc, _, _, _ = evaluate_model(type_model, type_val_loader)
    type_history['train_loss'].append(tr_loss); type_history['val_loss'].append(va_loss)
    type_history['train_acc'].append(tr_acc);   type_history['val_acc'].append(va_acc)
    print(f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Val Loss: {va_loss:.4f} Acc: {va_acc:.4f}')
    if va_acc > best_type_acc:
        best_type_acc = va_acc
        type_model.save_pretrained('codebert_type')
        tokenizer.save_pretrained('codebert_type')
        print(f'  ✅ Saved best type model (val acc={va_acc:.4f})')

print(f'\nBest type classifier val accuracy: {best_type_acc:.4f}')

## Cell 12 — Evaluate bug type classifier

In [ ]:
type_model = RobertaForSequenceClassification.from_pretrained('codebert_type').to(device)
_, _, type_preds, type_labels_out, _ = evaluate_model(type_model, type_test_loader)

type_names = [BUG_TYPES[i] for i in range(8)]
print('='*55)
print('BUG TYPE CLASSIFIER — TEST SET RESULTS')
print('='*55)
print(classification_report(type_labels_out, type_preds, target_names=type_names))

cm_type = confusion_matrix(type_labels_out, type_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm_type, annot=True, fmt='d', cmap='Purples',
            xticklabels=type_names, yticklabels=type_names)
plt.title('Bug Type Classifier — Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig('type_confusion_matrix.png',dpi=150); plt.show()

## Cell 13 — Attention-based line highlighting
Uses CodeBERT's attention weights to identify *which lines* are most suspicious. This is an interpretability feature found in published research.

In [ ]:
import gc
from transformers import RobertaForSequenceClassification

# ✅ Clear whatever is on GPU first
torch.cuda.empty_cache(); gc.collect()

print("Reloading binary_model with eager attention from trained checkpoint...")
binary_model = RobertaForSequenceClassification.from_pretrained(
    'codebert_binary',
    attn_implementation="eager"
).to(device)

# ✅ Load type_model to CPU first, move to GPU only during inference
print("Loading type_model onto CPU (moved to GPU only during inference)...")
type_model = RobertaForSequenceClassification.from_pretrained(
    'codebert_type',
    attn_implementation="eager"
)  # ✅ intentionally NOT .to(device) yet — saves GPU memory

print("Done.")

def get_attention_highlights(code_str, model, tokenizer, top_k=3):
    model.eval()
    enc = tokenizer(
        code_str, max_length=MAX_LEN,          # ✅ uses MAX_LEN not hardcoded 256
        padding='max_length', truncation=True, return_tensors='pt'
    )
    ids  = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)

    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=mask, output_attentions=True)

    if out.attentions is None:
        raise RuntimeError(
            "output_attentions returned None — make sure the model was loaded "
            "with attn_implementation='eager'"
        )

    last_attn = out.attentions[-1].squeeze(0).mean(0)[0].cpu().numpy()
    tokens    = tokenizer.convert_ids_to_tokens(ids[0].cpu())
    lines     = code_str.split('\n')

    line_scores = defaultdict(float)
    line_counts = defaultdict(int)
    line_idx = 0

    for pos, tok in enumerate(tokens):
        if tok in (tokenizer.pad_token, tokenizer.eos_token, tokenizer.bos_token):
            continue
        if tok == 'Ċ':
            line_idx = min(line_idx + 1, len(lines) - 1)
        if pos < len(last_attn):
            line_scores[line_idx] += float(last_attn[pos])
            line_counts[line_idx] += 1

    for li in line_scores:
        if line_counts[li] > 0:
            line_scores[li] /= line_counts[li]

    ranked = sorted(line_scores.items(), key=lambda x: x[1], reverse=True)
    return [(li, lines[li], score) for li, score in ranked[:top_k] if li < len(lines)]


sample_code = X_test[next(i for i, l in enumerate(y_test) if l == 1)]
highlights  = get_attention_highlights(sample_code, binary_model, tokenizer)

print('Top suspicious lines (by attention):')
for li, lt, sc in highlights:
    print(f'  Line {li:2d} (score={sc:.5f}): {lt.strip()}')

## Cell 14 — Full inference pipeline

In [ ]:
def full_analysis(code_str, threshold=None):
    # Use auto-computed threshold if none given
    t = threshold if threshold is not None else OPTIMAL_THRESHOLD  # ✅ from Cell 10

    binary_model.eval(); type_model.eval()
    enc  = tokenizer(code_str, max_length=256, padding='max_length',
                     truncation=True, return_tensors='pt')
    ids  = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)

    with torch.no_grad():
        bin_out  = binary_model(input_ids=ids, attention_mask=mask)
        type_out = type_model(input_ids=ids,   attention_mask=mask)

    bin_probs  = F.softmax(bin_out.logits,  dim=1)[0]
    type_probs = F.softmax(type_out.logits, dim=1)[0]
    is_buggy   = bin_probs[1].item() > t                # ✅ uses optimal threshold
    bug_prob   = bin_probs[1].item()

    if is_buggy:
        type_pred = type_probs[1:].argmax().item() + 1  # skip class 0 when buggy
    else:
        type_pred = 0

    type_conf  = type_probs[type_pred].item()
    highlights = get_attention_highlights(code_str, binary_model, tokenizer)

    return {
        'is_buggy':            is_buggy,
        'bug_probability':     bug_prob,
        'bug_type_name':       BUG_TYPES[type_pred] if is_buggy else 'clean',
        'bug_type_confidence': type_conf,
        'bug_description':     BUG_DESCRIPTIONS.get(type_pred, '') if is_buggy else 'No bug detected',
        'fix_suggestion':      BUG_FIXES.get(type_pred, 'Review logic.') if is_buggy else 'Code looks clean.',
        'suspicious_lines':    highlights,
    }


def print_analysis(r, code_str):
    bar = '█' * int(r['bug_probability'] * 40) + '░' * (40 - int(r['bug_probability'] * 40))
    print('=' * 60)
    print('VERDICT:', 'BUG DETECTED' if r['is_buggy'] else 'CLEAN')
    print('=' * 60)
    print(f"Bug probability  : {r['bug_probability']:.1%}  [{bar}]")
    print(f"Detection threshold: {OPTIMAL_THRESHOLD}")
    if r['is_buggy']:
        print(f"Bug type         : {r['bug_type_name']} (conf: {r['bug_type_confidence']:.1%})")
        print(f"Description      : {r['bug_description']}")
        print(f"Fix suggestion   : {r['fix_suggestion']}")
        if r['suspicious_lines']:
            print('\nSuspicious lines:')
            for li, lt, sc in r['suspicious_lines']:
                print(f'  Line {li+1:2d}: {lt.strip()}')
    else:
        print("No bug detected — code looks clean.")
    print('=' * 60)


print('--- Testing on a BUGGY sample ---')
buggy_sample = X_test[next(i for i, l in enumerate(y_test) if l == 1)]
print_analysis(full_analysis(buggy_sample), buggy_sample)

print('\n--- Testing on a CLEAN sample ---')
clean_sample = X_test[next(i for i, l in enumerate(y_test) if l == 0)]
print_analysis(full_analysis(clean_sample), clean_sample)

# ── Quick sanity check on your typo example ───────────────────────────────
print('\n--- Testing: returnn typo (should be BUG) ---')
typo_code = '''def add(a, b):
    result = a + b
    returnn result'''
print_analysis(full_analysis(typo_code), typo_code)

## Cell 15 — Research results dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14,10))

ep  = range(1, BINARY_EPOCHS+1)
ep2 = range(1, TYPE_EPOCHS+1)

axes[0,0].plot(ep, bin_history['train_acc'],'o-',label='Train',color='steelblue')
axes[0,0].plot(ep, bin_history['val_acc'],  's-',label='Val',  color='coral')
axes[0,0].set_title('Binary Model — Accuracy'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(ep2, type_history['train_acc'],'o-',label='Train',color='purple')
axes[0,1].plot(ep2, type_history['val_acc'],  's-',label='Val',  color='orange')
axes[0,1].set_title('Type Classifier — Accuracy'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

tc = Counter(bt_train)
tnames = [BUG_TYPES[i] for i in sorted(tc)]
tvals  = [tc[i] for i in sorted(tc)]
colors = ['#2ecc71','#e74c3c','#e67e22','#3498db','#9b59b6','#1abc9c','#f39c12','#c0392b']
axes[1,0].bar(tnames, tvals, color=colors)
axes[1,0].set_title('Bug Type Distribution')
axes[1,0].tick_params(axis='x', rotation=45)

buggy_p = [p for p,l in zip(test_probs, test_labels_out) if l==1]
clean_p = [p for p,l in zip(test_probs, test_labels_out) if l==0]
axes[1,1].hist(clean_p, bins=40, alpha=0.7, color='green', label='Clean')
axes[1,1].hist(buggy_p, bins=40, alpha=0.7, color='red',   label='Buggy')
axes[1,1].axvline(OPTIMAL_THRESHOLD, color='black', linestyle='--',
                  label=f'Threshold ({OPTIMAL_THRESHOLD})')
axes[1,1].set_title('Predicted Bug Probability Distribution')
axes[1,1].legend()

plt.suptitle('Research Bug Detection System — Results Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('research_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: research_dashboard.png')

## Cell 16 — Gradio interactive demo
Launches a shareable web UI. The public URL is printed below after launch.

In [ ]:
import gradio as gr

DEMO_CLEAN = '''def calculate_average(numbers):
    if not numbers:
        return 0.0
    total = sum(numbers)
    count = len(numbers)
    return total / count'''

DEMO_BUGGY = '''def calculate_average(numbers):
    if not numbers:
        return 0.0
    total = sum(numbers)
    count = len(numbers)
    return total / count + 1  # BUG: wrong arithmetic'''

DEMO_BUGGY2 = '''def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] != target:  # BUG: should be ==
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1'''

THRESHOLD = OPTIMAL_THRESHOLD   # ✅ uses auto-computed value from Cell 10                                   # ✅ single source of truth

def gradio_analyze(code_str):
    if not code_str.strip():
        return 'Please enter Python code.', '', '', '', ''
    r = full_analysis(code_str, threshold=THRESHOLD)  # ✅ passes threshold
    bar = '█' * int(r['bug_probability'] * 30) + '░' * (30 - int(r['bug_probability'] * 30))
    verdict = ('🐛  BUG DETECTED' if r['is_buggy'] else '✅  LOOKS CLEAN') + \
              f"\n\nBug probability  : {r['bug_probability']:.1%}\n" + \
              f"Clean probability: {1 - r['bug_probability']:.1%}\n[{bar}]"
    btype = (f"Type       : {r['bug_type_name']}\nConfidence : {r['bug_type_confidence']:.1%}\n"
             f"Description: {r['bug_description']}") if r['is_buggy'] else 'No bug type — code is clean.'
    fix = r['fix_suggestion']
    lines_txt = ''
    for li, lt, sc in r['suspicious_lines']:
        lines_txt += f'Line {li+1:2d} (attn={sc:.5f}): {lt.strip()}\n'
    info = (f'Binary model accuracy : {accuracy_score(test_labels_out, test_preds):.4f}\n'
            f'Detection threshold   : {THRESHOLD}\n'             # ✅ shows threshold in UI
            f'Training samples      : {len(X_train):,} real GitHub functions\n'
            f'Bug type classes      : {len(BUG_TYPES)}\n'
            f'Model                 : microsoft/codebert-base')
    return verdict, btype, fix, lines_txt, info

with gr.Blocks(title='Research Bug Detector', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🔬 Python Code Bug Detection System')
    gr.Markdown('Trained on **real GitHub functions**. Detects bugs, classifies type, highlights suspicious lines.')
    with gr.Row():
        with gr.Column(scale=2):
            code_box = gr.Code(label='Python code', language='python', value=DEMO_CLEAN, lines=18)
            btn = gr.Button('🔍  Analyze Code', variant='primary', size='lg')
        with gr.Column(scale=3):
            v_out = gr.Textbox(label='Verdict',          lines=5)
            t_out = gr.Textbox(label='Bug Type',         lines=4)
            f_out = gr.Textbox(label='Fix Suggestion',   lines=3)
            l_out = gr.Textbox(label='Suspicious Lines', lines=5)
            i_out = gr.Textbox(label='Model Info',       lines=4)
    gr.Examples([[DEMO_CLEAN], [DEMO_BUGGY], [DEMO_BUGGY2]], inputs=[code_box])
    btn.click(gradio_analyze, inputs=[code_box], outputs=[v_out, t_out, f_out, l_out, i_out])

demo.launch(share=True)

## Optional — Save models to Google Drive
Run this after training to keep your models between Colab sessions.
```python
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copytree('codebert_binary', '/content/drive/MyDrive/bug_detector/binary')
shutil.copytree('codebert_type',   '/content/drive/MyDrive/bug_detector/type')
print('Saved to Google Drive!')
```